# Before You Start

## What is an LLM Agent?

> ### **Common LLM Limitations:**
1. **Hallucination**
   - LLMs can generate incorrect information with high confidence
<center>
<img src="https://i.ibb.co/xSCvT7HM/agent-pr1.png" alt="agent pr1" border="0">
</center>

>2. **Knowledge Cutoff**
   - LLMs are limited to information from their training data
   - They cannot access or learn from information after their training period

<center>
<img src="https://i.ibb.co/3YQgC50m/agent-pr2.png" alt="agent pr2" border="0">
</center>

>3. **Data Privacy**
   - LLMs can only access public training data
   - They cannot access proprietary or private information

<center>
<img src="https://i.ibb.co/qZX4Dkf/agent-pr3.png" alt="agent pr3" border="0">
</center>

> ### **The Solution: LLM Agents**
An LLM Agent enhances a basic LLM by combining three key components:
- **LLM**: The core language model
- **Tools**: External capabilities and functions
- **Memory**: Ability to store and recall information

<center>
<img src="https://i.ibb.co/ycdMJt6r/agent.png" alt="agent" border="0" height="500">
</center>


> This combination helps overcome the limitations by:
- Using tools to verify information and access external data
- Maintaining memory of past interactions and information
- Enabling access to current data through external tools

## How Does it Work?

> The LLM agent follows a structured decision-making process when responding to instructions. This process consists of three main steps that repeat until the task is complete:
>1. **Think/Planning:** The agent analyzes the user's request and formulates a step-by-step plan to accomplish the task.
>2. **Action:** The agent executes specific actions by calling appropriate tools with the necessary parameters.
>3. **Observation:** The agent evaluates the results of its actions and determines the next steps based on the outcomes.

<center>
<img src="https://i.ibb.co/h1YJf6FT/agent2.png" alt="agent2" border="0" height="500">
</center>

>This cycle continues until the agent successfully completes the requested task or determines it cannot proceed further.

# Build Your First Agent Using LangGraph

> Let's build a sample agent that can:
* **Search the internet for information**
* **Do math calculations (+,-,*,/)**
* **Check the weather of a city**
* **arxiv paper info**

<center>
<img src="https://i.ibb.co/Mk5zzv1x/agent3.png" alt="agent3" border="0">
</center>

#### 1. install langchain library

In [ ]:
! uv pip install -U langgraph
! uv pip install langchain_community ddgs # ddgs for duckducksearch engine
! uv pip install langchain_groq # groq as llm provider

Using Python 3.12.13 environment at: /usr
Resolved 32 packages in 236ms
Checked 32 packages in 0.71ms
Using Python 3.12.13 environment at: /usr
Checked 2 packages in 88ms


* set env for agent tracing with [langsmith](https://smith.langchain.com)

In [ ]:
import os

# Set environment variables in the Python process so LangChain can access them
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "your_langsmit_api_key_xxxxxxxxx"
os.environ["LANGSMITH_PROJECT"] = "hack_ai"

print("LangSmith tracing environment variables set.")

LangSmith tracing environment variables set.


#### 2. Select Your LLM Model

<center>
<img src="https://i.ibb.co/k2b4Cv4j/agent4.png" alt="agent4" border="0" height="300">
</center>

>`langchain` supports multiple LLM providers including OpenAI, Google, and HuggingFace. For this tutorial, we'll use OpenAI's OSS `openai/gpt-oss-120b` model through their free API at **Groq**.

>To get started:
> 1. Visit [Groq Console](https://console.groq.com/keys)
2. Create an account if you don't have one
3. Generate an API key
4. Keep your API key secure - we'll use it in the next step.


In [ ]:
from langchain_groq import ChatGroq
# load model
MODEL = "openai/gpt-oss-120b"
GROQ_API_KEY = "XXXXXXXXXXXXX"
model = ChatGroq(
    model=MODEL,
    api_key = GROQ_API_KEY
)

In [ ]:
response = model.invoke(
    [
        {
            "role": "user",
            "content": "What is the capital of Morocco?",
        }
    ]
).content
response

'The capital of Morocco is **Rabat**.'

#### 3. Define Tools

>langchain supports multiple tool types:
- Build in LangChain [tools](https://docs.langchain.com/oss/python/integrations/tools)
- MCP (Model Control Protocol) tools
- Building your own one

>In this example, we'll demonstrate how to:
1. Create custom tools (using `@tool`)
2. Use predefined tools from `langchain`

>**Find out more langchain** [tools](https://docs.langchain.com/oss/python/integrations/tools).

<center>
<img src="https://i.ibb.co/cXrXzBH6/agent5.png" alt="agent5" border="0" height="500">
</center>

In [ ]:
from langchain.tools import tool
@tool
def calculator_tool(expression:str) -> str:
  # the tool should have a clear name. and a string documnetation.
  # string documentation has description of the tool and its inputs.
  "A calculator that can perform basic arithmetic operations"
  try:
      result = eval(expression)
      return str(result)
  except Exception as e:
      return f"Error: {str(e)}"

# test tool
print(calculator_tool.run("2 + 2 * 3"))

8


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
@tool
def city_weather(city: str) -> str:
    # the tool should have a clear name. and a string documnetation.
    # string documentation has description of the tool and its inputs.
    """
    Get the weather for a given city.
    Args:
        city: The name of the city.
    """
    # In a real-world scenario, this function would call a weather API.
    # For this example, we'll just return a dummy response.
    return f"The weather in {city} is sunny with a high of 25°C."

# test city_weather
tool_output = city_weather.run("Benguerir")
print(tool_output)

The weather in Benguerir is sunny with a high of 25°C.


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# Create a DuckDuckGo search tool
search_tool = DuckDuckGoSearchRun()
# Test Tool
tool_output = search_tool.invoke("What is the capital of Morocco?")
print(tool_output)

Morocco,[c] officially the Kingdom of Morocco,[d] is a country in the Maghreb region of North Africa.The culture of Morocco is a mix of Arab, Berber, European (specifically Andalusian[17]), and African cultures. Its capital is Rabat, while its largest city is Casablanca.[18]. Join me as I travel and vlog in the capital city of Morocco, Rabat. I walk through the beautiful Andalusian Gardens, explore the unbelievable Rabat Kasbah, a... Rabat is the capital and one of Morocco’s four imperial cities and the Moroccan government seat, housing the royal palace, the parliament, and several embassies, underscoring its role in the nation’s governance. The capital of Morocco is Rabat, which is not the most populated city in the country. It is located where the Bou Regreg River meets the Atlantic Ocean. Its population is approximately 577,827; the entire metro area has a population of around 1.2 million. The capital of Morocco is Rabat, located in the North African continent. It serves as the poli

In [ ]:
! uv pip install arxiv

Using Python 3.12.13 environment at: /usr
Checked 1 package in 97ms


In [ ]:
from langchain_community.tools.arxiv.tool import ArxivQueryRun

arxiv_tool = ArxivQueryRun()
tool_output = arxiv_tool.run("1706.03762")
print(tool_output)

Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new sing

#### 4. Integrate Tools: Create Your Agent

In [ ]:
from langchain.agents import create_agent

tools=[city_weather,search_tool,calculator_tool,arxiv_tool]
agent=create_agent(
    model=model,
    tools=tools
)

#### 5. Run Your Agent

In [ ]:
query="""What's the paper 1706.03762 about,
and who is Noam Shazeer?
Also, what is the weather in Benguerir?
And what is 2 + 2 * 3?"""
response=agent.invoke(
    {
    "messages": [{"role": "user", "content": query}]}
)

In [ ]:
print(response["messages"][-1].content)

**Paper 1706.03762 – “Attention Is All You Need”**  
*Published:* June 2017 (arXiv:1706.03762)  
*Authors:* Ashish Vaswani, **Noam Shazeer**, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  

**What it’s about**  
The paper introduces the **Transformer** architecture, a novel neural‑network design for sequence‑to‑sequence tasks (e.g., machine translation) that relies **solely on attention mechanisms** and eliminates recurrent or convolutional layers. Key points:

| Aspect | Description |
|--------|-------------|
| **Core idea** | Use multi‑head self‑attention to relate all positions in a sequence to each other in parallel. |
| **Benefits** | - Much higher parallelism (faster training). <br> - State‑of‑the‑art translation quality (BLEU scores > 28 on English‑German, > 41 on English‑French). <br> - Fewer parameters and lower computational cost compared with LSTM/CNN‑based models. |
| **Architecture** | Encoder‑decoder stacks of identical layers

#### 6. Add Memory

- If you want your agent to use memory you should to add short term memory with `InMemorySaver`, and also the `thread_id` is what keeps the conversation memory connected across calls, each new conversation will have a `thread_id`` .

<center>
<img src="https://i.ibb.co/Nd7cxpTC/agent7.png" alt="agent7" border="0" height="300">
</center>

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Short-term memory storage
memory = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[city_weather, search_tool, calculator_tool, arxiv_tool],
    checkpointer=memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "user_1" # set your id here
    }
}

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "hello my name abdeljalil"}
        ]
    },
    config=config
)
print(response["messages"][-1].content)

Hello, Abdeljalil! Nice to meet you. How can I help you today?


- by activating the memory the agent remember my name

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "what is my name?"}
        ]
    },
    config=config
)
print(response["messages"][-1].content)

Your name is **Abdeljalil**.


# See Agent Traces

After setting the langsmith variable above, you'll find the traces in [langsmith](https://smith.langchain.com)

* **example:** https://smith.langchain.com/public/ef878f04-9627-44fd-be6c-19a8de83faa7/r

# Exercice